In [28]:
# filed detection
import cv2
from imageio.core.imopen import imopen
from numpy.ma.core import shape
import numpy as np

# img = cv2.imread('../tmp/v5_bacground.png', cv2.IMREAD_COLOR)  # road.png is the filename

input_video_path = "../materials_part1/2024-11-18 17-53-16.mkv"  #"F:\\Videos\\2024-11-18 17-59-05.avi"  #"F:\\Videos\\2024-11-18 17-49-57.mkv"  #"F:\\Videos\\2024-11-18 17-59-05.avi"

cap = cv2.VideoCapture(input_video_path)

from utils.cut_image import get_cut_frame_from_frame

ret, image = cap.read()
if not ret:
    pass
image = get_cut_frame_from_frame(image, 2)
image = cv2.rotate(image, cv2.ROTATE_180)
img_tmp = cv2.blur(image, (3, 3))
cv2.imwrite("../tmp/hehe4.png", img_tmp)
from sklearn.cluster import KMeans


# flat = img.flatten()
def multiply_list(data):
    tmp = 1
    for i in data:
        tmp *= i
    return tmp


def cluster_and_recolor(img, n_clusters, mode: int = 1, top_n=None, binary=False):
    flat = img.reshape((multiply_list(img.shape) // 3, 3))

    kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init=3).fit(flat)

    cluster_color_centers = list()
    for i in kmeans.cluster_centers_:
        clr = list()
        for j in i:
            clr.append(int(j))
        cluster_color_centers.append(clr)
    cluster_color_centers = np.array(cluster_color_centers)
    # print(img, "hehe", flat, type(img), "hehe", cluster_color_centers)

    unflat = kmeans.labels_.reshape(img.shape[:-1]).astype(np.uint8)

    if mode == 1:
        return unflat * (255 // n_clusters)
    elif mode == 2 or mode == 3:
        sizes = [0] * len(cluster_color_centers)
        for i in kmeans.labels_:
            sizes[i] += 1
        tmp_sizes = list()
        for num, i in enumerate(sizes):
            tmp_sizes.append((-i, num))
        tmp_sizes = sorted(tmp_sizes)
        remap = [0] * len(sizes)
        for num, i in enumerate(tmp_sizes):
            remap[i[1]] = num
        if mode == 3:
            start_clr = cluster_color_centers[tmp_sizes[0][1]]
            size = tmp_sizes[0][0]
            if top_n is None:
                top_n = 0
            seen = [tmp_sizes[0][1]]
            for i in range(top_n):
                nearest = 0
                dist = np.inf
                size_tmp = 0
                for j in range(n_clusters):
                    if tmp_sizes[j][1] not in seen and np.linalg.norm(
                            start_clr - cluster_color_centers[tmp_sizes[j][1]]) < dist:
                        dist = np.linalg.norm(start_clr - cluster_color_centers[tmp_sizes[j][1]])
                        nearest = tmp_sizes[j][1]
                        size_tmp = tmp_sizes[j][0]
                seen.append(nearest)
                #start_clr = size * (start_clr / (size_tmp + size)) + cluster_color_centers[nearest] * (size_tmp / (size_tmp + size))
                size += size_tmp
            for j in seen:
                remap[j] = seen[0]
        for i in range(len(unflat)):
            for j in range(len(unflat[i])):
                if top_n is None or (mode == 3 and not binary):
                    unflat[i][j] = remap[unflat[i][j]]
                elif binary and mode == 3:
                    if remap[unflat[i][j]] == remap[tmp_sizes[0][1]]:
                        unflat[i][j] = 255
                    else:
                        unflat[i][j] = 0
                elif top_n > remap[unflat[i][j]]:
                    unflat[i][j] = 255
                else:
                    unflat[i][j] = 0
        if top_n is None or mode == 3:
            return unflat * (255 // n_clusters)
        else:
            return unflat
    elif mode == 4:
        for i in range(img.shape[0]):
            for j in range(img.shape[1]):
                img[i][j] = cluster_color_centers[unflat[i][j]]
        return img


image = cv2.blur(image, (3, 3))
# image = cv2.blur(image, (3, 3))
img = cluster_and_recolor(image, 13, 3, top_n=2, binary=True)  #13 3 2 // 30 3 6 // 20 3 3
# print(img)
# img = cluster_and_recolor(img,4)
# cv2.imshow('lanes', img)
# cv2.waitKey(0)
# cv2.destroyAllWindows()
#lines detection
from utils.image_tuner import tune_test, tune_from_params
import numpy as np
import math
import sklearn.cluster as skcl
from scipy.spatial import ConvexHull
from sklearn.linear_model import LinearRegression

import cv2

# image_t = tune_test("../tmp/hehe4.png")  #69 0 153 191 248 255
img_tmp = cv2.imread("../tmp/hehe4.png")
image_t = tune_from_params(img_tmp, 33, 0, 143, 181, 76, 255)
# cv2.imshow('lanes', image_t)
# cv2.waitKey(0)
# cv2.destroyAllWindows()
# print(img)
result_edges = image_t * 0
print()
for i in range(1, len(image_t) - 1):
    for j in range(1, len(image_t[i]) - 1):
        seen_white = 0
        seen_green = 0
        bad = 0
        for k in (-1, 0, 1):
            for t in (-1, 0, 1):
                if img[i + k][j + t] > 200:
                    seen_green += 1
                if image_t[i + k][j + t] > 200:
                    seen_white += 1
                if img[i + k][j + t] <= 200 and image_t[i + k][j + t] <= 200:
                    bad += 1
        if seen_white > 1 and seen_green > 2:
            # print("lane")
            result_edges[i][j] = 255

analysis = cv2.connectedComponentsWithStats(result_edges,
                                            4,
                                            cv2.CV_32S)
(totalLabels, label_ids, values, centroid) = analysis
areas = list()
for i in values:
    areas.append(i[cv2.CC_STAT_AREA])
areas = sorted(areas, reverse=True)
output = np.zeros(result_edges.shape, dtype="uint8")
for i in range(1, totalLabels):
    area = values[i, cv2.CC_STAT_AREA]

    if area >= areas[30]:
        # Labels stores all the IDs of the components on the each pixel
        # It has the same dimension as the threshold
        # So we'll check the component
        # then convert it to 255 value to mark it white
        componentMask = (label_ids == i).astype("uint8") * 255

        # Creating the Final output mask
        output = cv2.bitwise_or(output, componentMask)
    # print (analysis)


segments = cv2.HoughLinesP(output, 1.0, math.pi / 180.0, 80, 30, 10.0)
# print(segments.shape,segments)
segments = segments.reshape((len(segments), 2, 2))

# segments[i] = sorted(segments[i])
lines = list()
if False:  #clusterization method   #todo redo segments from vec<int,4> to vec<vec<int,2>,2>
    segm_by_direction_and_sp = list()
    for i_tmp in segments:
        i = i_tmp[0]
        x = i[2] - i[0]
        y = i[3] - i[1]
        if True:
            len_xy = math.sqrt(x * x + y * y)
            x /= len_xy
            y /= len_xy
        if x < 0:
            x *= -1
            y *= -1
        zero = (0, 0)
        if abs(x) > abs(y):
            c1 = i[0] / x
            zero = (0, i[1] - y * c1)
        else:
            c1 = i[1] / y
            zero = (i[0] - x * c1, 0)
        segm_by_direction_and_sp.append((x * 1000, y * 1000, zero[0], zero[1]))
    segm_by_direction_and_sp = np.array(segm_by_direction_and_sp)
    # clusters = skcl.OPTICS(min_samples=4).fit(segm_by_direction_and_sp)
    # clusters = skcl.DBSCAN(min_samples=2).fit(segm_by_direction_and_sp)
    clusters = skcl.HDBSCAN(min_samples=2).fit(segm_by_direction_and_sp)
    # clusters = KMeans(n_clusters=10).fit(segm_by_direction_and_sp)
    centers_of_clusters = dict()
    for i in range(len(segm_by_direction_and_sp)):
        if clusters.labels_[i] not in centers_of_clusters.keys():
            centers_of_clusters[clusters.labels_[i]] = [np.linalg.norm(segments[i][0][:1] - segments[i][0][2:])]
            # print(type(segm_by_direction_and_sp[i].tolist()),,centers_of_clusters[clusters.labels_[i]])
            centers_of_clusters[clusters.labels_[i]].extend(
                (segm_by_direction_and_sp[i] * np.linalg.norm(segments[i][0][:1] - segments[i][0][2:])).tolist())
        else:
            centers_of_clusters[clusters.labels_[i]][0] += np.linalg.norm(segments[i][0][:1] - segments[i][0][2:])
            for j in range(len(segm_by_direction_and_sp[i])):
                centers_of_clusters[clusters.labels_[i]][j + 1] += segm_by_direction_and_sp[i][j] * np.linalg.norm(
                    segments[i][0][:1] - segments[i][0][2:])
    for i in centers_of_clusters.values():
        # print(i)
        # i = [1]
        # i.extend(i_tmp)
        lines.append([[int((i[-2] - 1000 * i[1]) / i[0]), int((i[-1] - 1000 * i[2]) / i[0]),
                       int((i[-2] + 1000 * i[1]) / i[0]), int((i[-1] + 1000 * i[2]) / i[0])]])

#idea: if 2 segments are on line then area on 4 points is minimal


if False:
    for i in range(len(segments)):  #sorted so segms are from x1 to x2 where x1<x2 or x1==x2 and y1<y2
        if segments[i][0][0] < segments[i][1][0]:
            segments[i] = np.array([segments[i][1], segments[i][0]])
        elif segments[i][0][0] == segments[i][1][0] and segments[i][0][1] < segments[i][1][1]:
            segments[i] = np.array([segments[i][1], segments[i][0]])

    st = set()
    for i in segments:
        st.add(tuple(i.reshape((4)).tolist()))
    while True:
        # concatinating segments by using best start and best end points while area metric allows it
        first_best = second_best = None
        best_area = 0
        for i_tmp in st:
            for j_tmp in st:
                i = np.array(i_tmp)
                j = np.array(j_tmp)
                if (i != j).any():
                    area = 0
                    points = np.array([i, j]).reshape((4, 2))
                    for i in range(len(points)):
                        area += np.cross(points[i], points[(i + 1) % len(points)])

                    if first_best is None:
                        first_best = i
                        second_best = j
                        best_area = area
                    else:
                        if best_area > area:
                            best_area = area
                            first_best = i
                            second_best = j
        if area > 500:
            break

        points = np.array([first_best, second_best]).reshape((2, 4))
        line = LinearRegression().fit(points[0], points[1])
        start = (0, line.intercept_)
        direction = (1, line.coef_)


        def project_point(point, start_line, direction_line):
            point -= start_line
            sk = 0
            len_line = 0
            for i in range(len(line)):
                len_line += direction_line[i] ** 2
                sk += direction_line[i] * point[i]
            sk /= len_line
            return start_line + direction_line * sk


        result = (0, 0)
        if first_best[0] < second_best[0]:
            result[0] = first_best[0]
        else:
            result[0] = second_best[0]
        if first_best[0] > second_best[0]:
            result[1] = first_best[1]
        else:
            result[1] = second_best[1]
        result[0] = project_point(result[0], start, direction)
        result[1] = project_point(result[1], start, direction)
        st.remove(first_best)
        st.remove(second_best)
        st.add(np.array(result))
    lines = list(st)

#idea 3: same as idea 2, but using this metric in dbscan
labels = 0
n_clusters = 0
if True:

    def custom_distance_v1(a: np.array, b: np.array):  # works iff a and b goes into different directions
        a = a.reshape((2, 2))
        b = b.reshape((2, 2))
        points = [a[0], a[1], b[0], b[1]]
        sum = np.cross(points[-1], points[0])
        for i in range(1, len(points)):
            sum += np.cross(points[i - 1], points[i])
        return np.linalg.norm(sum)  #eps = 10


    def custom_distance_v2(a: np.array, b: np.array):
        a = a.reshape((2, 2))
        b = b.reshape((2, 2))
        points = [a[0], a[1], b[0], b[1]]
        sum = np.cross(points[-1], points[0])
        for i in range(1, len(points)):
            sum += np.cross(points[i - 1], points[i])
        points[0], points[1] = points[1], points[0]
        sum2 = np.cross(points[-1], points[0])
        for i in range(1, len(points)):
            sum2 += np.cross(points[i - 1], points[i])
        return max(np.linalg.norm(sum), np.linalg.norm(sum2))  #eps = 100


    def custom_distance_v3(a: np.array, b: np.array):
        a = a.reshape((2, 2))
        b = b.reshape((2, 2))
        points = [a[0], a[1], b[0], b[1]]
        sum = np.cross(points[-1], points[0])
        for i in range(1, len(points)):
            sum += np.cross(points[i - 1], points[i])
        points[0], points[1] = points[1], points[0]
        sum2 = np.cross(points[-1], points[0])
        for i in range(1, len(points)):
            sum2 += np.cross(points[i - 1], points[i])
        return max(np.linalg.norm(sum), np.linalg.norm(sum2)) / (
                    np.linalg.norm(a[0] - a[1]) + np.linalg.norm(b[0] - b[1]))  #eps = 5


    # print(segments)
    clustering = skcl.DBSCAN(eps=5, min_samples=1, metric=custom_distance_v3).fit(
        segments.reshape((segments.shape[0], 4)))
    labels = clustering.labels_
    n_clusters = len(np.unique(labels)) - (1 if -1 in labels else 0)
    # print("labels,cluster_n ",labels,n_clusters)


def add_up_segments_v1(data: np.array):
    data = data.astype(np.float64)

    #prepare
    # def is_same_direction(a,b):
    #     v1 = np.cross(b-a[0],a[1]-a[0])/np.linalg.norm(a[1]-a[0])*(a[1]-a[0]).reshape((1,2))
    #     delta = np.linalg.norm((a[1]-a[0])-(v1[1]-v1[0]))
    #     return delta<1e-5
    # def is_same_direction_v2(a,b):
    #     v1 = np.cross(b-a[0],a[1]-a[0])/np.linalg.norm(a[1]-a[0])
    #     delta = 1-(v1[1]-v1[0])
    #     return delta<1e-5
    def is_same_direction_v3(a, b):
        return np.dot(a[1] - a[0], b[1] - b[0]) > 0

    # print(is_same_direction_v3(np.array([[1,2],[3,4]]),np.array([[9,10],[7,8]])))#false
    # print(is_same_direction_v3(np.array([[1,2],[3,4]]),np.array([[5,6],[7,8]])))#true
    for i in range(1, len(data)):
        if not is_same_direction_v3(data[0], data[i]):
            data[i][0], data[i][1] = data[i][1], data[i][0]
    sum_weight = np.linalg.norm(data[0][1] - data[0][0])
    sum_direction = data[0][1] - data[0][0]
    avg_point = (data[0][0] + data[0][1]) * sum_weight
    for i in range(1, len(data)):
        tmp_w = np.linalg.norm(data[i][1] - data[i][0])
        sum_direction += data[i][1] - data[i][0]
        avg_point += (data[i][0] + data[i][1]) * tmp_w
        sum_weight += tmp_w
    sum_direction /= sum_weight
    avg_point /= sum_weight * 2
    start = 0
    end = 0
    for i in range(len(data)):
        start = min(start, np.dot(sum_direction, data[i][0] - avg_point))
        end = max(end, np.dot(sum_direction, data[i][1] - avg_point))
    return (np.array([start * sum_direction, end * sum_direction]) + avg_point).astype(np.int32)


grouped = dict()
for i in range(len(segments)):
    if labels[i] not in grouped.keys():
        grouped[labels[i]] = [segments[i]]
    else:
        grouped[labels[i]].append(segments[i])
new_segments = list()
for i in grouped.values():
    new_segments.append(add_up_segments_v1(np.array(i)))
segments = new_segments


def homography_matrix(segs):
    def homography_matrix_by_4_vec(vecs):  # vecs = [a,b,c,d],  a,b -> (1,0) c,d ->(0,1)
        for i in range(len(vecs)):
            vecs[i] = np.append(vecs[i], 1)
        # notice that a-b = alpha is eigenvector with value 0 as well as c-d = beta
        # than:
        # A = V*L*V^(-1)
        # L = [[0,0,0],[0,0,0],[0,0,x]]
        # and V = [alpha,beta,alpha cross beta]
        alpha = vecs[0] - vecs[1]
        beta = vecs[2] - vecs[3]
        gamma = np.cross(alpha, beta)

    # idea v2: try all matricies and choose one which maximizes metric.
    # for all  segments generate vectors length of 1
    # for all pairs of this vectors find matrix that minimizes some function
    vectors = list()
    for i in segs:
        point = (i[0] - i[1]) / np.linalg.norm(i[0] - i[1])
        vectors.append(point)
    vectors_v2 = np.array(vectors).reshape((len(vectors), 1, 2))
    # def get_metric(vecs,)
    best_val = -1e9
    best_m = np.identity(3)
    for i in range(len(vectors)):
        for j in range(len(vectors)):
            if i != j:
                for k in range(len(vectors)):
                    if i != k and j != k:
                        for f in range(len(vectors)):
                            if f != k and f != j and f != k:
                                mat, mask = cv2.findHomography(
                                    np.array([vectors[f], vectors[i], vectors[j], vectors[k]]),
                                    np.array([[0, 100], [100, 0], [100, 0], [0, 100]]))
                                # print(vectors.shape,mat.shape)
                                vecs = cv2.perspectiveTransform(vectors_v2, mat).reshape((len(vectors), 2))
                                groups_size = [0, 0, 0]  # hor,vert,bad
                                for t in vecs:
                                    t_p = t / np.linalg.norm(t)
                                    if np.abs(np.abs(np.dot(t_p, np.array([1, 0]))) - 1) < 1 / 8:
                                        groups_size[0] += 1
                                    elif np.abs(np.abs(np.dot(t_p, np.array([0, 1]))) - 1) < 1 / 8:
                                        groups_size[1] += 1
                                    else:
                                        groups_size[2] += 1
                                val = groups_size[0] * groups_size[1] - groups_size[2]
                                # print(groups_size)
                                if val > best_val:
                                    print(groups_size)
                                    # print()
                                    best_val = val
                                    best_m = mat

    print(best_val, best_m)

    return best_m



# homograpy matrix, for transfromation that minimizes sum of distances between each end of segments and line that goes trough center of segment and "horizontal" or "vertical" point at the horizon, where this points represents points to which are all parallel horizontal/vertical lines are converging
def homography_matrix_v2(edges: list):
    #hereinafter assuming center of image is (0,0), i.e. (0,0) is point through perpendicular from "focal point" is goes
    #todo:convert edges to p1 and p2 solution: watch in deepseek, has intersting way of searching with ransak. also points p1 and p2 are called vanishing points and there are some papers in internet about them.
    def find_distance_fp_to_plane_v1(fov_pixels,
                                     field_of_view):  #hor[0]*x+hor[1]*y = hor[2] ; field_of_view in degrees - angle at which edges of image displayed, fov_pixels is number of pixels corresponding to fov
        return fov_pixels / (2 * math.tan(field_of_view / 2))

    # do not work if horizontal lines are parallel to horizon.
    #assuming that perpendicular line from "focal point" (point from which rays are shoot in point-and-plane approximation of camera) onto "focal plane" (plane of image in approximation ...) goes through center of image
    def find_distance_fp_to_plane_v2(p1, p2, p3,
                                     angle=math.pi / 2) -> float:  #p1 is horizont point for vertical lines,p2 is horizont point for horizontal lines, p3 is nearest point on horizon to center of image. returns distance in pixels
        #let's derive formula:
        # look on triangle FP, p1,p2:
        # let segment p1-p3 called a, p3-p2 called b, angle fp-p1-p2 is beta, angle p2-fp-p1 is alpha, segment fp-p3 is h
        # angle fp-p3-p1 is 90 deg based on assumption
        # then tg(beta) = h/a, tg(180-alpha-beta) = -tg(alpha + beta) = h/b
        #then -tg(beta)/tg(alpha+beta) = b/a
        #then -tg(beta)/((tg(alpha)+tg(beta))/(tg(1-tg(alpha)tg(beta))) = b/a
        #then tg(beta) = x, tg(alpha) = c
        # (-x+c*x**2)/(x+c) = b/a
        #then -x+c*x**2 -x*b/a-c*b/a = 0
        # same as: c*x**2 +(-1-b/a)*x-c*b/a = 0
        # d = 1+2*b/a+(b/a)**2+4*c**2*b/a
        # x = ...: x is positive
        a = np.linalg.norm(p1 - p3)
        b = np.linalg.norm(p2 - p3)
        c_p = math.tan(
            math.pi / 2 - angle)  # c_p = ctg(angle) = tg(90-angle) = 1/tg(angle) needed because tg(90) -> inf
        # d = 1+2*b/a+(b/a)**2+4*b/a*c*c
        # x = ((1+b/a)+math.sqrt(d))/(2*c)
        x = ((1 + b / a) * c_p + math.sqrt((1 + 2 * b / a + (b / a) ** 2) * c_p ** 2 + 4 * b / a)) / 2
        h = x * a
        return h

    def calculate_p3(p1, p2):
        # a = np.linalg.norm(p1)
        # b = np.linalg.norm(p2)
        # c = np.linalg.norm(p1-p2)
        a_p = np.dot(p1, p2 - p1)  #same in case of relation as: np.dot(-p1,p2-p1)/c# same as: (np.dot(-p1,p2-p1)/a/c)*a
        b_p = np.dot(p2, p1 - p2)  #same in case of relation as: np.dot(-p2,p1-p2)/c# same as: (np.dot(-p2,p1-p2)/b/c)*b
        return a_p / (a_p + b_p) * p1 + b_p / (a_p + b_p) * p2

    def angle_of_camera_v1(distance_to_fp, p3):
        return math.atan(np.linalg.norm(p3) / distance_to_fp)

    def homography_matrix_internal(p1, angle, scale, p3, distance_to_fp):
        """
        assuming camera axis lower than horizon :todo fix this
        :param p1:  is horizont point for vertical lines
        :param angle: angle of tilt perpendicular to horizon
        :param scale: relation of real field(in pixels) to pixels on image
        :param p3: p3
        :param distance_to_fp: one extensive parameter, distance to focal point
        :return: homography matrix
        """
        hdir = np.array([-p3[1], p3[0]])
        hdir /= np.linalg.norm(hdir)
        hp1 = hdir * 1000
        hp2 = -hdir * 1000
        hpr1 = hp1 * scale
        hpr2 = hp2 * scale
        p3_normed = p3/np.linalg.norm(p3)
        pvert_scale = 1 / (
                    2 * math.cos(angle / 2))  #equals: math.sin(angle/2)/math.sin(angle)# assuming angle less than pi/2
        hp3 = p3 * pvert_scale
        hpr3 = distance_to_fp * scale * p3_normed
        hp4 = -math.tan(angle) * distance_to_fp * p3_normed
        hpr4 = -p3_normed * math.sin(angle) * scale * distance_to_fp
        #decomposition onto orthogonal basis
        p1_p_hdir = np.dot(hdir, p1) * hdir
        p1_p_p3 = p1 - p1_p_hdir
        #projection of basis
        p1_p_p3r = p3_normed * scale / (
                    math.sin(angle) * (distance_to_fp / np.linalg.norm(p1_p_p3)) - math.cos(angle))  # some geometry
        p1_p_hdirr = p1_p_hdir * scale * math.sin(angle) / math.sin(
            angle - math.atan(np.linalg.norm(p1_p_p3) / distance_to_fp))
        p1r =  p1_p_p3r+p1_p_hdirr #  decompose vector into hp3 and hdir, project hdir with scale2(where scale2 computed from sinus theorem and scale), and hp3 with scale of (smth)

        cos_rot = np.dot(p1r,np.array((1,0)))/np.linalg.norm(p1r)
        sin_rot = math.sqrt(1-cos_rot**2)
        rot_mat = np.array([[cos_rot,sin_rot],[-sin_rot,cos_rot]])# todo maybe wrong direction

        hpr1 = rot_mat.dot(hpr1)
        hpr2 = rot_mat.dot(hpr2)
        hpr3 = rot_mat.dot(hpr3)
        hpr4 = rot_mat.dot(hpr4)

        return cv2.findHomography(np.array([hp1,hp2,hp3,hp4]),np.array([hpr1,hpr2,hpr3,hpr4]))

# matrix = homography_matrix(segments)

# print(image_t)


/tmp/ipykernel_6871/3627590980.py:331: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  sum = np.cross(points[-1], points[0])
/tmp/ipykernel_6871/3627590980.py:333: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  sum += np.cross(points[i - 1], points[i])
/tmp/ipykernel_6871/3627590980.py:335: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  sum2 = np.cross(points[-1], points[0])
/tmp/ipykernel_6871/3627590980.py:337: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  sum2 += np.cross(points[i - 1], points[i])
/tmp/ipykernel_6871/3627590980.py:331: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays 

In [29]:
import copy


print(output)

output_2 = cv2.cvtColor(copy.deepcopy(output), cv2.COLOR_GRAY2BGR)
print(output_2.shape)

output_2 //= 5
for i in range(len(segments)):
    print(segments[i][0], segments[i][1])
    output_2 = cv2.line(output_2, segments[i][0], segments[i][1], (0, 0, 255) ) # 40 kmeans is good

# cv2.imshow('lanes', output_2)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
(537, 714, 3)
[  1 505] [712 472]
[  0 444] [710 418]
[505  66] [713 374]
[195 375] [500 365]
[211 386] [522 378]
[138 167] [563 168]
[  0 405] [126  94]
[149  43] [404  47]
[ 20 251] [123  34]
[210  34] [429  35]
[505  78] [700 386]
[ 20 409] [152  47]
[238  59] [414  61]
[627 202] [709 315]
[711 469] [711 314]
[186 246] [234 254]
[361  87] [447  94]
[435 107] [478 119]


In [ ]:
import cv2
import numpy as np

def line_center(line):
    """Вычисление центра линии (среднее двух концов)."""
    return np.array([(line[0][0] + line[1][0]) / 2, (line[0][1] + line[1][1]) / 2])

def line_angle(line):
    """Вычисление угла линии в градусах относительно горизонтали."""
    (x1, y1), (x2, y2) = line
    angle = np.degrees(np.arctan2((y2 - y1), (x2 - x1)))
    return angle

def average_line(lines):
    """Вычисляет представительную (усреднённую) линию для группы линий."""
    x1s = [line[0][0] for line in lines]
    y1s = [line[0][1] for line in lines]
    x2s = [line[1][0] for line in lines]
    y2s = [line[1][1] for line in lines]
    avg_line = [[np.mean(x1s), np.mean(y1s)], [np.mean(x2s), np.mean(y2s)]]
    return avg_line

def intersection(line1, line2):
    """Находит точку пересечения двух линий (каждая задана двумя точками)."""
    x1, y1 = line1[0]
    x2, y2 = line1[1]
    x3, y3 = line2[0]
    x4, y4 = line2[1]
    denom = (x1 - x2)*(y3 - y4) - (y1 - y2)*(x3 - x4)
    if denom == 0:
        return None
    px = ((x1*y2 - y1*x2)*(x3 - x4) - (x1 - x2)*(x3*y4 - y3*x4)) / denom
    py = ((x1*y2 - y1*x2)*(y3 - y4) - (y1 - y2)*(x3*y4 - y3*x4)) / denom
    return [px, py]

def draw_line(img, line, color):
        pt1 = tuple(map(int, line[0]))
        pt2 = tuple(map(int, line[1]))
        cv2.line(img, pt1, pt2, color, 2)

def process_segments(segments, img_shape, output_img):
    """
    Обрабатывает сегменты (линии) с учётом перспективы.

    :param segments: список линий в виде [[[x1,y1],[x2,y2]], ...]
    :param img_shape: кортеж (высота, ширина) изображения (например, output_2.shape[:2])
    :param output_img: изображение для отрисовки результатов (копия исходного)
    :return: результат с отрисованным эллипсом, а также словарь с промежуточными данными
    """

    # Следующие 4 переменные - коэффициенты прямоугольника, линии, центры которых внутри будут удалены из общего списка

    upper_compression_coef_h_inner = 0.7     # Нижняя граница прямоугольника
    lower_compression_coef_h_inner = 0.15    # Верхняя граница прямоугольника
    upper_compression_coef_w_inner = 0.8   # левая граница прямоугольника
    lower_compression_coef_w_inner = 0.2   # правая граница прямоугольника

    # Следующие 4 переменные - коэффициенты прямоугольника, но будут удалены линии, центры которых будут вне прямоугольника

    upper_compression_coef_h_outer = 0.95     # Нижняя граница прямоугольника
    lower_compression_coef_h_outer = 0.05    # Верхняя граница прямоугольника
    upper_compression_coef_w_outer = 0.95  # левая граница прямоугольника
    lower_compression_coef_w_outer = 0.05   # правая граница прямоугольника

    height, width = img_shape

    # 1. Фильтрация: отсекаем линии, центр которых находится в прямоугольнике
    # Идея: можно попробовать найти центр, а потом удалить линии, что расположены в некоторм проценте от центра
    x_center_min_inner, x_center_max_inner = width * lower_compression_coef_w_inner, width * upper_compression_coef_w_inner
    y_center_min_inner, y_center_max_inner = height * lower_compression_coef_h_inner, height * upper_compression_coef_h_inner
    x_center_min_outer, x_center_max_outer = width * lower_compression_coef_w_outer, width * upper_compression_coef_w_outer
    y_center_min_outer, y_center_max_outer = height * lower_compression_coef_h_outer, height * upper_compression_coef_h_outer
    filtered = []
    for seg in segments:
        c = line_center(seg)
        if  not (x_center_min_inner < c[0] < x_center_max_inner and y_center_min_inner < c[1] < y_center_max_inner):
            if  not ((x_center_min_outer > c[0] or c[0] > x_center_max_outer) or (y_center_min_outer > c[1] or c[1]> y_center_max_outer)):
                filtered.append(seg)
                # output_img = cv2.line(output_img, seg[0], seg[1], (255, 255, 0) ) # отрисовка полученных линий
    # отрисовка прямоугольника
    # output_img = cv2.rectangle(output_img, np.array((x_center_min_inner, y_center_min_inner), dtype=np.int32), np.array((x_center_max_inner, y_center_max_inner), dtype=np.int32), (255, 255, 255), 2)
    # output_img = cv2.rectangle(output_img, np.array((x_center_min_outer, y_center_min_outer), dtype=np.int32), np.array((x_center_max_outer, y_center_max_outer), dtype=np.int32), (255, 180, 0), 2)
    # cv2.imshow('lanes1', output_img)
    # cv2.waitKey(0)
    # cv2.destroyAllWindows()





    # 2. Разделяем линии на горизонтальные и вертикальные по углу наклона.
    horizontal, vertical = [], []
    threshold_h = 30
    threshold_v = 40
    for seg in filtered:
        angle = line_angle(seg)
        # горизонтальные: угол близок к 0 или 180 градусов (с допуском ±30)
        if abs(angle) < threshold_h or abs(abs(angle) - 180) < threshold_h:
            horizontal.append(seg)
            # output_img = cv2.line(output_img, seg[0], seg[1], (255, 0, 0), 2) # debug
        # вертикальные: угол около ±90 градусов (допуск от 60 до 120)
        elif 90 - threshold_v < abs(angle) < 90 + threshold_v:
            vertical.append(seg)
            # output_img = cv2.line(output_img, seg[0], seg[1], (0, 255, 0), 2) # debug
        else:
            print(abs(angle))
    # Отобразить линии, разделенные на 2 линии
    # cv2.imshow('lanes1', output_img)
    # cv2.waitKey(0)
    # cv2.destroyAllWindows()

    # Если недостаточно линий, завершаем.
    if len(horizontal) < 2 or len(vertical) < 2:
        print("Недостаточно линий для разделения на группы.")
        return None

    # 3. Кластеризация: горизонтальные разделяем по оси Y, вертикальные — по оси X.
    horizontal_sorted = sorted(horizontal, key=lambda seg: line_center(seg)[1])
    mid_h = len(horizontal_sorted) // 2
    near_horizontal = horizontal_sorted[:mid_h]  # верхние (меньшие y)
    far_horizontal  = horizontal_sorted[mid_h:]  # нижние (большие y)

    # Debug: вывод ближних\дальних горизонтальных
    # for seg in near_horizontal:
    #     output_img = cv2.line(output_img, seg[0], seg[1], (255, 0, 0), 2)
    # for seg in far_horizontal:
    #     output_img = cv2.line(output_img, seg[0], seg[1], (0, 255, 0), 2)
    # cv2.imshow('lanes1', output_img)
    # cv2.waitKey(0)
    # cv2.destroyAllWindows()

    vertical_sorted = sorted(vertical, key=lambda seg: line_center(seg)[0])
    mid_v = len(vertical_sorted) // 2
    left_vertical = vertical_sorted[:mid_v]
    right_vertical = vertical_sorted[mid_v:]

    # Debug: print left\right line
    # for seg in left_vertical:
    #     output_img = cv2.line(output_img, seg[0], seg[1], (255, 0, 0), 2)
    # for seg in right_vertical:
    #     output_img = cv2.line(output_img, seg[0], seg[1], (0, 255, 0), 2)
    # cv2.imshow('lanes1', output_img)
    # cv2.waitKey(0)
    # cv2.destroyAllWindows()

    # 4. Представительные линии: усредняем каждую группу.
    rep_near = average_line(near_horizontal)
    rep_far  = average_line(far_horizontal)
    rep_left = average_line(left_vertical)
    rep_right= average_line(right_vertical)

    draw_line(output_img, rep_near, (0, 255, 0))
    draw_line(output_img, rep_far, (0, 255, 0))
    draw_line(output_img, rep_left, (255, 0, 0))
    draw_line(output_img, rep_right, (255, 0, 0))

    # cv2.imshow('lanes1', output_img)
    # cv2.waitKey(0)
    # cv2.destroyAllWindows()

    # 5. Находим пересечения представительных линий (вершины четырёхугольника).
    top_left = intersection(rep_near, rep_left)
    top_right = intersection(rep_near, rep_right)
    bottom_left = intersection(rep_far, rep_left)
    bottom_right = intersection(rep_far, rep_right)

    quad = [top_left, top_right, bottom_right, bottom_left]  # порядок важен

    # Проверяем, что все вершины вычислены.
    if any(pt is None for pt in quad):
        print("Не удалось вычислить все пересечения.")
        return None

    # 6. Вычисляем размеры целевого прямоугольника.
    def distance(p, q):
        return np.linalg.norm(np.array(p) - np.array(q))

    width_top = distance(top_left, top_right)
    width_bottom = distance(bottom_left, bottom_right)
    width_target = int(max(width_top, width_bottom))

    height_left = distance(top_left, bottom_left)
    height_right = distance(top_right, bottom_right)
    height_target = int(max(height_left, height_right))

    # Выпрямленный прямоугольник
    dst_rect = np.array([
        [0, 0],
        [width_target - 1, 0],
        [width_target - 1, height_target - 1],
        [0, height_target - 1]
    ], dtype="float32")
    # Прямоугольник в перспективе
    src_quad = np.array(quad, dtype="float32")

    # 7. Вычисляем матрицу перспективного преобразования и её обратную.
    M = cv2.getPerspectiveTransform(src_quad, dst_rect)
    Minv = cv2.getPerspectiveTransform(dst_rect, src_quad)

    # 8. В прямоугольнике вписываем эллипс. Для максимальной площади эллипс,
    # вписанный в прямоугольник, имеет центр в центре и полуоси равные половине размеров.
    center_rect = (width_target / 2, height_target / 2)
    axes_rect = (width_target / 2, height_target / 2)
    angle_rect = 0  # оси параллельны сторонам прямоугольника

    # 9. Генерируем точки контура эллипса в координатах прямоугольника.
    num_points = 500
    t = np.linspace(0, 2*np.pi, num_points)
    ellipse_points = np.zeros((num_points, 1, 2), dtype="float32")
    for i in range(num_points):
        x = center_rect[0] + axes_rect[0] * np.cos(t[i])
        y = center_rect[1] + axes_rect[1] * np.sin(t[i])
        ellipse_points[i, 0] = [x, y]
    ellipse_center = np.zeros((1, 1, 2), dtype="float32")
    ellipse_center[0, 0] = [center_rect[0], center_rect[1]]

    # 10. Обратным перспективным преобразованием получаем точки эллипса в исходной системе координат.
    ellipse_points_transformed = cv2.perspectiveTransform(ellipse_points, Minv)
    ellipse_points_transformed = np.int32(ellipse_points_transformed.reshape(-1, 2))
    # И центр
    ellipse_center = cv2.perspectiveTransform(ellipse_center, Minv)
    ellipse_center = np.int32(ellipse_center.reshape(-1, 2))

    # 11. Отрисовка результатов:
    # Отрисуем представительные линии (для наглядности)

    draw_line(output_img, rep_near, (0, 255, 0))
    draw_line(output_img, rep_far, (0, 255, 0))
    draw_line(output_img, rep_left, (255, 0, 0))
    draw_line(output_img, rep_right, (255, 0, 0))

    # Отрисуем вершины четырёхугольника
    for pt in quad:
        cv2.circle(output_img, tuple(map(int, pt)), 5, (0, 0, 255), -1)

    # Отрисуем эллипс, полученный обратным преобразованием
    cv2.polylines(output_img, [ellipse_points_transformed], isClosed=True, color=(0, 255, 255), thickness=2)
    # И его центр
    print(ellipse_center)
    cv2.circle(output_img, ellipse_center[0], 5, (0, 255, 0), -1)

    # Для отладки: warped изображение (прямоугольник)
    warped = cv2.warpPerspective(output_img, M, (width_target, height_target))
    cv2.ellipse(warped, (int(center_rect[0]), int(center_rect[1])), (int(axes_rect[0]), int(axes_rect[1])), angle_rect, 0, 360, (255, 0, 255), 2)

    return output_img, warped, {
        "rep_near": rep_near,
        "rep_far": rep_far,
        "rep_left": rep_left,
        "rep_right": rep_right,
        "quad": quad,
        "M": M,
        "Minv": Minv,
        "dst_rect": dst_rect,
        "ellipse_points_rect": ellipse_points,
        "ellipse_points_transformed": ellipse_points_transformed
    }

if __name__ == "__main__":
    img_shape = output_2.shape[:2]
    output_img = output_2.copy()

    result = process_segments(segments, img_shape, output_img)
    if result:
        output_with_ellipse, warped, data = result
        # Показываем результаты (если запускается локально)
        cv2.imshow("Original with ellipse", output_with_ellipse)
        # cv2.imshow("Warped (rectangular domain)", warped) # Debug
        cv2.waitKey(0)
        cv2.destroyAllWindows()


[[342 169]]
